<a href="https://colab.research.google.com/github/gabrielmprata/Info_seguranca_publica_rj/blob/main/Info_seguranca_publica_rj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img loading="lazy" src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/python/python-original.svg" width="40" height="40"/> <img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/pandas/pandas-original-wordmark.svg" width="40" height="40"/>   <img loading="lazy" src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/plotly/plotly-original-wordmark.svg" width="40" height="40"/>

---
**Pré Processamento de dados**
>
**Dev**: Gabriel Prata
>
**Data**: 01/03/2026
>
**Última modificação**: 01/03/2026
>
**Contexto**: *Dados abertos de segurança pública*
>
---

#**<font color=#4c60d6 size="6"> Import libraries**

In [1]:
# Importação de pacotes
import pandas as pd
import numpy as np
import missingno as ms # para tratamento de missings
import datetime
import re # expressão regulares
from random import randint # gerar inteiros aliatórios

#bibliotecas para visualização de dados
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt

#importar a biblioteca para usar a Distribuição Binomial de Bernoulli
from scipy.stats import bernoulli

#compactar
from shutil import copyfileobj
import bz2

# Configuração para não exibir os warnings
import warnings
warnings.filterwarnings("ignore")

#**<font color=#4c60d6 size="6"> 1. Coleta de dados**

In [3]:
# importando dataset

# URL de importação

url   = "https://raw.githubusercontent.com/gabrielmprata/Info_seguranca_publica_rj/main/datasets/BaseDPEvolucaoMensalCisp.csv.bz2"

df_ocorrencias = pd.read_csv(url, compression='bz2', encoding = "Latin 1", delimiter=';')

#**<font color=#4c60d6 size="6"> 2. Análise de Dados Inicial**

###**<font color=#4c60d6> 2.1. Estatísticas Descritivas**

Compreende a organização, o resumo e, descrever os dados, que podem ser expressos em tabelas e gráficos.
>
Veremos a seguir alguns comandos para exibir algumas estatísticas descritivas.
>
---

In [4]:
#	Quantidade de atributos e instâncias (linhas/colunas)
df_ocorrencias.shape

(37314, 65)

<font color=#4c60d6> Data frame com 65 atributos(features) e 37.314 tuplas.


---



In [5]:
# Exibir os 5 primeiros registros
df_ocorrencias.head(5)

,cisp,mes,ano,mes_ano,aisp,risp,munic,mcirc,regiao,hom_doloso,...,cmp,cmba,ameaca,pessoas_desaparecidas,encontro_cadaver,encontro_ossada,pol_militares_mortos_serv,pol_civis_mortos_serv,registro_ocorrencias,fase
0,1,1,2003,2003m01,5,1,Rio de Janeiro,3304557,Capital,0,...,NaN,NaN,21,2,0,0,0,0,578,3
1,4,1,2003,2003m01,5,1,Rio de Janeiro,3304557,Capital,3,...,NaN,NaN,15,6,0,1,0,0,441,3
2,5,1,2003,2003m01,5,1,Rio de Janeiro,3304557,Capital,3,...,NaN,NaN,47,2,1,0,0,0,637,3
3,6,1,2003,2003m01,1,1,Rio de Janeiro,3304557,Capital,6,...,NaN,NaN,26,2,1,0,0,0,473,3
4,7,1,2003,2003m01,1,1,Rio de Janeiro,3304557,Capital,4,...,NaN,NaN,10,1,3,0,0,0,147,3




---



In [6]:
# Mostra diversas informações do Dataframe em um único comando, e exibir o uso de memória
df_ocorrencias.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37314 entries, 0 to 37313
Data columns (total 65 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   cisp                        37314 non-null  int64  
 1   mes                         37314 non-null  int64  
 2   ano                         37314 non-null  int64  
 3   mes_ano                     37314 non-null  object 
 4   aisp                        37314 non-null  int64  
 5   risp                        37314 non-null  int64  
 6   munic                       37314 non-null  object 
 7   mcirc                       37314 non-null  int64  
 8   regiao                      37314 non-null  object 
 9   hom_doloso                  37314 non-null  int64  
 10  lesao_corp_morte            37314 non-null  int64  
 11  latrocinio                  37314 non-null  int64  
 12  cvli                        37314 non-null  int64  
 13  hom_por_interv_policial     373

<font color=#4c60d6> Data frame utilizando 24.2 MB de memória.


---

In [7]:
# Quantidade de valores únicos
df_ocorrencias.nunique()

,0
cisp,138
mes,12
ano,24
mes_ano,277
aisp,42
...,...
encontro_ossada,8
pol_militares_mortos_serv,5
pol_civis_mortos_serv,3
registro_ocorrencias,2004




---



In [9]:
# Quantidade de NaN/Missing/Nulls no dataframe
df_ocorrencias.isnull().sum()

,0
cisp,0
mes,0
ano,0
mes_ano,0
aisp,0
...,...
encontro_ossada,0
pol_militares_mortos_serv,0
pol_civis_mortos_serv,0
registro_ocorrencias,0




---



###**<font color=#4c60d6> 2.2. Distribuição dos atributos**

>Nessa etapa, iremos verificar a distribuição dos principais atributos. Para ver se existe a necessidade de tomar alguma ação de transformações na etapa de preparação de dados.


---

In [10]:
df_ocorrencias.describe().round(2)

,cisp,mes,ano,aisp,risp,mcirc,hom_doloso,lesao_corp_morte,latrocinio,cvli,...,cmp,cmba,ameaca,pessoas_desaparecidas,encontro_cadaver,encontro_ossada,pol_militares_mortos_serv,pol_civis_mortos_serv,registro_ocorrencias,fase
count,37314.00,37314.00,37314.00,37314.00,37314.00,37314.00,37314.00,37314.00,37314.00,37314.00,...,32668.00,32668.00,37314.00,37314.00,37314.00,37314.00,37314.00,37314.00,37314.00,37314.00
mean,81.72,6.48,2014.19,20.15,3.87,3791939.01,2.93,0.03,0.10,3.05,...,8.74,0.64,41.31,3.22,0.36,0.02,0.01,0.00,444.01,2.99
std,48.32,3.46,6.63,11.35,2.01,1741131.78,4.33,0.18,0.37,4.44,...,14.81,1.64,39.39,4.16,0.99,0.18,0.13,0.05,398.54,0.12
min,1.00,1.00,2003.00,1.00,1.00,3300100.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2.00
25%,38.00,3.00,2008.00,10.00,2.00,3302601.00,0.00,0.00,0.00,0.00,...,2.00,0.00,14.00,0.00,0.00,0.00,0.00,0.00,106.00,3.00
50%,78.00,6.00,2014.00,20.00,4.00,3304557.00,1.00,0.00,0.00,1.00,...,5.00,0.00,30.00,2.00,0.00,0.00,0.00,0.00,370.00,3.00
75%,125.00,9.00,2020.00,30.00,6.00,3304557.00,4.00,0.00,0.00,4.00,...,11.00,1.00,56.00,5.00,0.00,0.00,0.00,0.00,663.00,3.00
max,168.00,12.00,2026.00,43.00,7.00,9999999.00,43.00,6.00,9.00,43.00,...,419.00,60.00,370.00,191.00,112.00,10.00,4.00,2.00,3185.00,3.00
